# Wykluczenia

**Pytanie:** którzy filerzy ruszają ETF-ami JPMorgan z powodów innych niż decyzja klienta, o którą zespół sprzedaży mógłby powalczyć?

**Wniosek:** wykluczamy sam JPMorgan, animatorów rynku i dwóch filerów z błędami w danych. Duzi dystrybutorzy (Morgan Stanley, LPL, Bank of America, UBS) zostają, bo to prawdziwi klienci ETF. Lista: `config/exclusions.yaml`, uzasadnienie: `docs/decisions.md`.

Miara: **obrót** = średnio przehandlowane w kwartale / średnio trzymane. Zwykły zarządzający obraca kilkanaście procent pozycji na kwartał, animator rynku całą.

In [1]:
import duckdb
import yaml

con = duckdb.connect()
%load_ext sql
%sql con
%config SqlMagic.displaylimit = 40

house = yaml.safe_load(open("../config/house.yaml"))
excluded = [e["cik"] for e in yaml.safe_load(open("../config/exclusions.yaml"))["exclusions"]]
PREFIXES = tuple(house["cusip_prefixes"])
EXCLUDED = tuple(excluded)

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

## 1. Obrót na filera w ETF-ach JPMorgan

Zmiana akcji względem poprzedniego kwartału razy cena z bieżącego. Splitów nie korygujemy, bo ETF-y JPMorgan ich nie miały (`03_house_family.ipynb`). Całkowite wyjście z pozycji nie ma wiersza i nie liczy się do obrotu, co dla wykrycia animatorów rynku nie przeszkadza.

In [2]:
con.sql(f"""
CREATE OR REPLACE TABLE filers AS
WITH jpm AS (
    SELECT
        CIK, PERIODOFREPORT, CUSIP, shares, value_corrected,
        lag(shares) OVER (PARTITION BY CIK, CUSIP ORDER BY PERIODOFREPORT) AS prev_shares,
        lag(PERIODOFREPORT) OVER (PARTITION BY CIK, CUSIP ORDER BY PERIODOFREPORT) AS prev_period
    FROM '../data/processed/positions_pit.parquet'
    WHERE left(CUSIP, 6) IN {PREFIXES}
),
holdings AS (
    SELECT CIK, PERIODOFREPORT, sum(value_corrected) AS held_usd FROM jpm GROUP BY 1, 2
),
trades AS (
    SELECT CIK, PERIODOFREPORT,
           sum(abs(shares - prev_shares) * (value_corrected * 1.0 / shares)) AS traded_usd
    FROM jpm
    WHERE shares > 0 AND prev_shares IS NOT NULL
      AND PERIODOFREPORT = last_day(prev_period + INTERVAL 3 MONTH)
    GROUP BY 1, 2
),
per_filer AS (
    SELECT h.CIK,
           count(*) AS quarters,
           avg(h.held_usd) AS avg_held_usd,
           avg(coalesce(t.traded_usd, 0)) AS avg_traded_usd,
           avg(coalesce(t.traded_usd, 0)) / avg(h.held_usd) AS turnover
    FROM holdings h LEFT JOIN trades t USING (CIK, PERIODOFREPORT)
    GROUP BY h.CIK
),
names AS (
    SELECT lpad(s.CIK, 10, '0') AS CIK, mode(c.FILINGMANAGER_NAME) AS name
    FROM '../data/processed/submission/*.parquet' s
    JOIN '../data/processed/coverpage/*.parquet' c USING (ACCESSION_NUMBER)
    GROUP BY 1
)
SELECT p.*, n.name, p.CIK IN {EXCLUDED} AS excluded
FROM per_filer p LEFT JOIN names n USING (CIK)
""")

CountBIGINT
3848


Rozkład obrotu dla filerów trzymających średnio ponad 1 mln USD: mediana około 0,13, 99. percentyl około 0,8.

In [3]:
%%sql
SELECT count(*) AS filerzy,
       round(quantile_cont(turnover, 0.5), 2) AS mediana,
       round(quantile_cont(turnover, 0.9), 2) AS p90,
       round(quantile_cont(turnover, 0.99), 2) AS p99
FROM filers WHERE avg_held_usd > 1e6

Running query in 'DuckDBPyConnection'

filerzy,mediana,p90,p99
2574,0.13,0.32,0.8


## 2. Najwięksi pod względem obrotu w USD

Na górze są JPMorgan, duzi dystrybutorzy i Jane Street. Dystrybutorzy mają niski obrót (0,08–0,2), czyli trzymają, a nie obracają.

In [4]:
%%sql
SELECT CIK, name, quarters,
       round(avg_held_usd / 1e6, 1) AS trzyma_mln, round(avg_traded_usd / 1e6, 1) AS obraca_mln,
       round(turnover, 2) AS obrot, excluded
FROM filers ORDER BY avg_traded_usd DESC LIMIT 40

Running query in 'DuckDBPyConnection'

CIK,name,quarters,trzyma_mln,obraca_mln,obrot,excluded
0001834011,Legacy Financial Group LLC,10,5670.1,5047.2,0.89,True
0000019617,JPMORGAN CHASE & CO,15,48084.7,4468.8,0.09,True
0001600151,"Arete Wealth Advisors, LLC",15,2211.9,623.5,0.28,False
0001403438,LPL Financial LLC,15,3945.4,585.2,0.15,False
0000895421,MORGAN STANLEY,15,7125.5,582.0,0.08,False
0000070858,BANK OF AMERICA CORP /DE/,15,4777.9,505.1,0.11,False
0001595888,"JANE STREET GROUP, LLC",15,504.3,453.1,0.9,True
0001407543,ENVESTNET ASSET MANAGEMENT INC,15,1778.5,347.2,0.2,False
0000720005,RAYMOND JAMES FINANCIAL INC,7,2358.4,328.3,0.14,False
0001677044,"OSAIC HOLDINGS, INC.",15,1107.9,220.2,0.2,False


## 3. Animatorzy rynku i dealerzy po nazwie

Nazwy firm, o których wiadomo, że zarabiają na obsłudze handlu ETF-ami. Większość ma obrót wyraźnie powyżej mediany.

In [5]:
%%sql
SELECT CIK, name, quarters,
       round(avg_held_usd / 1e6, 1) AS trzyma_mln, round(avg_traded_usd / 1e6, 1) AS obraca_mln,
       round(turnover, 2) AS obrot, excluded
FROM filers
WHERE regexp_matches(upper(name), 'JANE STREET|SUSQUEHANNA|CITADEL|FLOW TRADERS|OLD MISSION|IMC|GTS|TWO SIGMA SEC|HRT|XTX|VIRTU FIN|SIMPLEX|WOLVERINE|OPTIVER|FINANCIAL MARKETS|SG AMERICAS|WORLD MARKET')
ORDER BY avg_traded_usd DESC

Running query in 'DuckDBPyConnection'

CIK,name,quarters,trzyma_mln,obraca_mln,obrot,excluded
0001595888,"JANE STREET GROUP, LLC",15,504.3,453.1,0.9,True
0001446194,"SUSQUEHANNA INTERNATIONAL GROUP, LLP",15,140.1,150.2,1.07,True
0001423053,CITADEL ADVISORS LLC,15,105.0,80.8,0.77,True
0001466697,FLOW TRADERS U.S. LLC,15,95.6,56.4,0.59,True
0001455915,OLD MISSION CAPITAL LLC,15,74.4,39.3,0.53,True
0001166588,BNP PARIBAS FINANCIAL MARKETS,14,17.3,12.9,0.75,True
0001452861,"IMC-Chicago, LLC",4,16.7,9.7,0.58,True
0001452765,GTS SECURITIES LLC,13,12.4,4.1,0.33,True
0001421224,CIBC WORLD MARKET INC.,15,8.0,2.4,0.3,True
0001450144,"TWO SIGMA SECURITIES, LLC",15,3.3,1.1,0.33,True


## 4. Błędy w danych

Legacy Financial Group ma portfel około 0,5 mld USD, ale w dwóch kwartałach raportuje ponad 400 mld, w tym 56 mld w ETF-ach JPMorgan. To wartości zawyżone około 1000 razy, których korekta jednostek nie obsługuje (łapie tylko zaniżone). Quattro trzyma około 2 mln USD, a „obraca” 200 mln na kwartał, czyli liczba akcji jest błędna.

In [6]:
%%sql
SELECT CIK, PERIODOFREPORT, count(*) AS pozycje,
       round(sum(value_corrected) / 1e9, 2) AS portfel_mld,
       round(sum(value_corrected) FILTER (WHERE left(CUSIP, 6) IN ('46641Q', '46654Q')) / 1e6, 1) AS jpm_mln
FROM '../data/processed/positions_pit.parquet'
WHERE CIK IN ('0001834011', '0001705929')
GROUP BY 1, 2 ORDER BY 1, 2

Running query in 'DuckDBPyConnection'

CIK,PERIODOFREPORT,pozycje,portfel_mld,jpm_mln
0001705929,2022-12-31,79,0.25,1.1
0001705929,2023-03-31,84,0.24,1.3
0001705929,2023-06-30,83,0.25,1.9
0001705929,2023-09-30,81,0.23,1.7
0001705929,2023-12-31,78,0.29,4.0
0001705929,2024-03-31,80,0.35,0.3
0001705929,2024-06-30,76,0.38,2.0
0001705929,2024-09-30,72,0.35,3.2
0001705929,2024-12-31,86,0.39,3.1
0001705929,2025-03-31,84,0.29,3.1


## 5. Ile pozytywów zabierają wykluczenia?

Pozytyw jak w `03_house_family.ipynb`: ponad 1 mln USD nowych pieniędzy w ETF-ach JPMorgan w kwartale t+1.

In [7]:
con.sql(f"""
CREATE OR REPLACE TABLE flows AS
WITH periods AS (
    SELECT DISTINCT CIK, PERIODOFREPORT AS t FROM '../data/processed/positions_pit.parquet'
    WHERE PERIODOFREPORT >= '2024-12-31'
),
pairs AS (
    SELECT a.CIK, a.t, b.t AS t1 FROM periods a
    JOIN periods b ON a.CIK = b.CIK AND b.t = last_day(a.t + INTERVAL 3 MONTH)
),
pos AS (
    SELECT CIK, PERIODOFREPORT AS t, CUSIP, shares, value_corrected AS v
    FROM '../data/processed/positions_pit.parquet'
    WHERE left(CUSIP, 6) IN {PREFIXES} AND PERIODOFREPORT >= '2024-12-31'
),
price AS (SELECT CUSIP, t, median(v * 1.0 / shares) AS px FROM pos WHERE shares > 0 AND v > 0 GROUP BY 1, 2),
a AS (SELECT p.*, pr.t1 FROM pos p JOIN pairs pr USING (CIK, t)),
b AS (SELECT p.CIK, pr.t, p.t AS t1, p.CUSIP, p.shares FROM pos p JOIN pairs pr ON p.CIK = pr.CIK AND p.t = pr.t1),
j AS (
    SELECT coalesce(a.CIK, b.CIK) AS CIK, coalesce(a.t, b.t) AS t, coalesce(a.t1, b.t1) AS t1,
           coalesce(a.CUSIP, b.CUSIP) AS CUSIP, coalesce(a.shares, 0) AS s0, coalesce(b.shares, 0) AS s1
    FROM a FULL JOIN b ON a.CIK = b.CIK AND a.t = b.t AND a.CUSIP = b.CUSIP
)
SELECT j.CIK, j.t, sum((j.s1 - j.s0) * px.px) AS new_money, j.CIK IN {EXCLUDED} AS excluded
FROM j JOIN price px ON px.CUSIP = j.CUSIP AND px.t = j.t1
GROUP BY j.CIK, j.t
""")

CountBIGINT
15031


In [8]:
%%sql
SELECT excluded,
       count(*) FILTER (WHERE new_money > 1e6) AS pozytywy,
       round(sum(new_money) FILTER (WHERE new_money > 1e6) / 1e9, 1) AS nowe_pieniadze_mld
FROM flows GROUP BY excluded ORDER BY excluded

Running query in 'DuckDBPyConnection'

excluded,pozytywy,nowe_pieniadze_mld
False,4006,76.6
True,47,12.0
